<a href="https://www.kaggle.com/code/bassetkerouche/fork-of-models-xgb-lgbm-cat-copie-from-cris?scriptVersionId=289670427" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# XGB Boosting Over Residuals - CV 0.05595
This is a starter notebook demonstrating "boosting over residuals". For the original dataset, @siukeitin (Kaggle user broccoli beef) published the optimal Bayesian solution [here][2]. (This is based on the original data's generation formula [here][1]).

For this month's playground competition, Kaggle created synthetic data from this original data. Therefore the signal has been altered and augmented. Hence the optimal Bayesian solution is no longer optimal for this month's playground competition's data.

None-the-less, we can begin with this optimal solution and have XGB learn how to improve it. So, instead of training with `target`, we will train our XGB with `target minus optimal solution` (i.e. the residual). XGB will learn to predict this residual aka "boost over residuals"!

Discussion about this notebook is [here][3]

[1]: https://www.kaggle.com/code/ianktoo/simulated-road-accident-data-generator
[2]: https://www.kaggle.com/competitions/playground-series-s5e10/discussion/609994#3296622
[3]: https://www.kaggle.com/competitions/playground-series-s5e10/discussion/610828

# Load Train, Test, Original

In [1]:
import pandas as pd, numpy as np

train = pd.read_csv("/kaggle/input/playground-series-s5e10/train.csv")
print("Train shape:", train.shape )
train.head()

Train shape: (517754, 14)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [2]:
test = pd.read_csv("/kaggle/input/playground-series-s5e10/test.csv")
test['accident_risk'] = 0.5
print("Test shape:", test.shape )
test.head()

Test shape: (172585, 14)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1,0.5
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0,0.5
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1,0.5
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2,0.5
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3,0.5


In [3]:
orig = []
for k in [2,10,100]:
    df = pd.read_csv(f"/kaggle/input/simulated-roads-accident-data/synthetic_road_accidents_{k}k.csv")
    orig.append(df)
orig = pd.concat(orig,axis=0)
orig['id'] = np.arange(len(orig))+test['id'].max()+1
orig = orig[ train.columns ] 
print("Original data shape:", orig.shape )
orig.head()

Original data shape: (112000, 14)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,690339,rural,2,0.72,60,daylight,clear,True,False,afternoon,False,False,2,0.37
1,690340,highway,4,0.95,45,daylight,foggy,False,True,evening,False,True,1,0.40
2,690341,rural,1,0.72,25,night,rainy,False,False,evening,True,False,1,0.55
3,690342,rural,4,0.86,70,dim,foggy,True,False,morning,True,True,1,0.56
4,690343,highway,1,0.00,60,night,rainy,True,True,morning,True,True,3,0.54


In [4]:
combine = pd.concat([train,test,orig],axis=0,ignore_index=True)
print("Combine shape:", combine.shape )
combine.head()

Combine shape: (802339, 14)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


# Feature Engineer
We will preprocess/feature engineer the following:
* Add @siukeitin optimal original data solution (from [here][1])
* Label encode the categorical features
* Target encode all features using original data targets

[1]: https://www.kaggle.com/competitions/playground-series-s5e10/discussion/609994#3296622

In [5]:
FEATURES = list( orig.columns[1:-1] )
TARGET = orig.columns[-1]
print(f"Features: {FEATURES}, Target: '{TARGET}'")

Features: ['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'time_of_day', 'holiday', 'school_season', 'num_reported_accidents'], Target: 'accident_risk'


## - Add New Feature!

In [6]:
# https://www.kaggle.com/competitions/playground-series-s5e10/discussion/609994#3296622
import scipy

def f(X):
    return \
    0.3 * X["curvature"] + \
    0.2 * (X["lighting"] == "night").astype(int) + \
    0.1 * (X["weather"] != "clear").astype(int) + \
    0.2 * (X["speed_limit"] >= 60).astype(int) + \
    0.1 * (X["num_reported_accidents"] > 2).astype(int)

def clip(f):
    def clip_f(X):
        sigma = 0.05
        mu = f(X)
        a, b = -mu/sigma, (1-mu)/sigma
        Phi_a, Phi_b = scipy.stats.norm.cdf(a), scipy.stats.norm.cdf(b)
        phi_a, phi_b = scipy.stats.norm.pdf(a), scipy.stats.norm.pdf(b)
        return mu*(Phi_b-Phi_a)+sigma*(phi_a-phi_b)+1-Phi_b
    return clip_f

z = clip(f)(combine)
combine["y"] = z.values
FEATURES.append("y")

## - Identify Nums and Cats

In [7]:
CATS = []
NUMS = []
for c in FEATURES:
    t = "CAT"
    if combine[c].dtype=='object':
        CATS.append(c)
    else:
        NUMS.append(c)
        t = "NUM"
    n = combine[c].nunique()
    na = combine[c].isna().sum()
    print(f"[{t}] {c} has {n} unique and {na} NA")
print("CATS:", CATS )
print("NUMS:", NUMS )

[CAT] road_type has 3 unique and 0 NA
[NUM] num_lanes has 4 unique and 0 NA
[NUM] curvature has 298 unique and 0 NA
[NUM] speed_limit has 5 unique and 0 NA
[CAT] lighting has 3 unique and 0 NA
[CAT] weather has 3 unique and 0 NA
[NUM] road_signs_present has 2 unique and 0 NA
[NUM] public_road has 2 unique and 0 NA
[CAT] time_of_day has 3 unique and 0 NA
[NUM] holiday has 2 unique and 0 NA
[NUM] school_season has 2 unique and 0 NA
[NUM] num_reported_accidents has 11 unique and 0 NA
[NUM] y has 1019 unique and 0 NA
CATS: ['road_type', 'lighting', 'weather', 'time_of_day']
NUMS: ['num_lanes', 'curvature', 'speed_limit', 'road_signs_present', 'public_road', 'holiday', 'school_season', 'num_reported_accidents', 'y']


## - Label Encode Cats

In [8]:
SIZES = {}
for c in CATS:
    combine[c],_ = combine[c].factorize()
    SIZES[c] = combine[c].max()+1
    combine[c] = combine[c].astype('int32')
    combine[c] = combine[c].astype('int32')
print("Cardinality of all CATS:", SIZES )

Cardinality of all CATS: {'road_type': 3, 'lighting': 3, 'weather': 3, 'time_of_day': 3}


In [9]:
col='y'
combine[f"{col}_dec1"] = ((combine[col] * 10) % 10).fillna(-1).astype(int)
# combine[f"{col}_dec2"] = ((combine[col] * 100) % 10).fillna(-1).astype(int)
# combine[f"{col}_dec4"] = ((combine[col] * 10000) % 10).fillna(-1).astype(int)

col='curvature'
combine[f"{col}_dec1"] = ((combine[col] * 10) % 10).fillna(-1).astype(int)
# col='speed_limit'
# 
# combine[f"{col}_bin1"] = (combine[col] // 10).fillna(-1).astype(int)

In [10]:
col ="lighting"
combine[f"{col}_log1p"] = np.log1p(combine[col])
col ="num_reported_accidents"
combine[f"{col}_log1p"] = np.log1p(combine[col])
col ="curvature"
combine[f"{col}_log1p"] = np.log1p(combine[col])
col ="speed_limit"
combine[f"{col}_log1p"] = np.log1p(combine[col])
col ="y"
combine[f"{col}_cbrt"] = np.cbrt(combine[col])
combine[f"{col}_log1p"] = np.log1p(combine[col])
combine[f"{col}_sqrt"] = np.sqrt(combine[col])
# col ="num_reported_accidents"
# combine[f"{col}_cbrt"] = np.cbrt(combine[col])
# combine[f"{col}_sqrt"] = np.sqrt(combine[col])

In [11]:
train = combine.iloc[:len(train)]
test = combine.iloc[len(train):len(train)+len(test)]
orig = combine.iloc[-len(orig):]
print(f"Train shape: {train.shape}, Test shape: {test.shape}, Original data shape: {orig.shape}")

Train shape: (517754, 24), Test shape: (172585, 24), Original data shape: (112000, 24)


## - Target Encode

In [12]:
TE = []
for c in FEATURES:
    tmp = orig.groupby(c)[TARGET].mean()
    n = f"TE_{c}"
    print(f"{n}, ",end="")
    tmp.name = n
    train = train.merge(tmp, on=c, how='left')
    test = test.merge(tmp, on=c, how='left')
    TE.append(n)

TE_road_type, TE_num_lanes, TE_curvature, TE_speed_limit, TE_lighting, TE_weather, TE_road_signs_present, TE_public_road, TE_time_of_day, TE_holiday, TE_school_season, TE_num_reported_accidents, TE_y, 

In [13]:
for c in ['curvature','speed_limit']:
    for i in range(-2,2):
        train[c+f"_{i}"]=(train[c]*(10**i)%10).astype(np.int8)
        test[c+f"_{i}"]=(test[c]*(10**i)%10).astype(np.int8)
        # orig[c+f"_{i}"]=(orig[c]*(10**i)%10).astype(np.int8)
        if train[c+f"_{i}"].nunique()==1:
            train.drop([c+f"_{i}"],axis=1,inplace=True)
            test.drop([c+f"_{i}"],axis=1,inplace=True)
            # orig.drop([c+f"_{i}"],axis=1,inplace=True)

In [14]:
BINARY_COLS=['road_signs_present','public_road','holiday','school_season']
for df in [train,test ]:
    df['BINARY']=0
    for i in range(len(BINARY_COLS)):
        df['BINARY']+=df[BINARY_COLS[i]].astype(int)*(2**i)

In [15]:
aggs = ['mean','max','min','nunique','count']

for c in FEATURES:
    tmp = (orig.groupby(c)[TARGET]           
                .agg(aggs)                         
                .rename(columns=lambda a: f'{c}_{TARGET}_{a}')  
                .reset_index())
    train = train.merge(tmp, on=c, how='left')
    test  = test.merge(tmp, on=c, how='left')

train.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,...,num_reported_accidents_accident_risk_mean,num_reported_accidents_accident_risk_max,num_reported_accidents_accident_risk_min,num_reported_accidents_accident_risk_nunique,num_reported_accidents_accident_risk_count,y_accident_risk_mean,y_accident_risk_max,y_accident_risk_min,y_accident_risk_nunique,y_accident_risk_count
0,0,0,2,0.06,35,0,0,False,True,0,...,0.362937,0.92,0.0,93,37520,0.117305,0.25,0.00,23.0,256.0
1,1,0,4,0.99,35,0,1,True,False,1,...,0.364480,0.96,0.0,95,24889,0.288561,0.46,0.15,25.0,132.0
2,2,1,4,0.63,70,1,1,False,True,2,...,0.363730,0.95,0.0,92,28162,0.393284,0.53,0.28,26.0,204.0
3,3,2,4,0.07,35,1,0,True,True,2,...,0.362937,0.92,0.0,93,37520,0.116632,0.25,0.00,26.0,288.0
4,4,1,1,0.58,60,0,2,False,False,1,...,0.362937,0.92,0.0,93,37520,0.474207,0.64,0.33,29.0,328.0


# Train XGBoost on Residuals
We will train XGB on residuals. Instead of using `accident_risk` as target, we will use `target = accident_risk - y` where `y` is @siukeitin (Kaggle user broccoli beef) optimal original data solution from [here][1]

[1]: https://www.kaggle.com/competitions/playground-series-s5e10/discussion/609994#3296622

In [16]:
import os, sys
from contextlib import contextmanager

@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout




In [17]:
params = {'batch_size': 'auto',
          'patience': 16,
          'allow_amp': True,
          'arch_type': 'tabm-mini',
          'tabm_k': 32,
          'gradient_clipping_norm': 1.0, 
          'share_training_batches': False,
          'lr': 0.0029993695720154537,
          'weight_decay': 0.023742083301699905,
          'n_blocks': 3,
          'd_block': 448, 
          'dropout': 0.0, 
          'num_emb_type': 'pwl',
          'd_embedding': 32,
          'num_emb_n_bins': 119,
         }

In [18]:
# oof_preds = np.zeros(len(X))
# test_preds = np.zeros(len(test))

# for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
#     print(f'--- Fold {fold+1}/{N_SPLITS} ---')
    
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#     with suppress_stdout():
#         model = TabM_D_Regressor(**params)
#         model.fit(X_train, y_train, X_val, y_val, cat_col_names=CATS)
    
#     oof_preds[val_idx] = model.predict(X_val)
#     test_preds += model.predict(X_test)

#     print(f"Fold {fold+1} RMSE: {root_mean_squared_error(y_val, oof_preds[val_idx]):.5f}")

# test_preds /= N_SPLITS

# print(f"Overall OOF RMSE: {root_mean_squared_error(y, oof_preds):.5f}")

In [19]:
from sklearn.model_selection import KFold
import xgboost as xgb

print(f"XGBoost version {xgb.__version__}")

XGBoost version 2.0.3


In [20]:
import matplotlib.pyplot as plt

In [21]:
FOLDS = 7
SEED = 42

params = {
    "objective": "reg:squarederror",   
    "eval_metric": "rmse",             
    "learning_rate": 0.01,
    "max_depth": 5,                    
    "subsample": 0.9279560370175621,
    "colsample_bytree": 0.5332228504482441,
    "seed": SEED,
    "device": "cuda",
}
# {'max_depth': 5, 'subsample': 0.9279560370175621, 'colsample_bytree': 0.5332228504482441}. Best is trial 33 with value: 0.055922729964457366.
 # {'max_depth': 6, 'subsample': 0.692874348545021, 'colsample_bytree': 0.5587449284285676}#. Best is trial 14 with value: 0.05592517858299127.
 # {'max_depth': 6, 'subsample': 0.7839364936400282,'colsample_bytree': 0.6496808171460874}#. Best is trial 20 with value: 0.05592445752023536.

In [22]:
FOLDS = 7
SEED = 42

params_l = {
    "objective": "regression",        # LightGBM's equivalent of reg:squarederror
    "metric": "rmse",                 # Evaluation metric
    "boosting_type": "gbdt",          # Standard gradient boosting
    "learning_rate": 0.01,
    "max_depth": 5,
    "num_leaves": 2**5 - 1,           # Good rule of thumb: (2^max_depth - 1)
    "subsample": 0.9279560370175621,  # Same as XGBoost subsample
    "colsample_bytree": 0.5332228504482441,
    "min_data_in_leaf": 20,           # Helps prevent overfitting
    "feature_fraction": 0.5332228504482441,  # alias for colsample_bytree
    "bagging_fraction": 0.9279560370175621,  # alias for subsample
    "bagging_freq": 1,
    "lambda_l1": 0.0,                 # L1 regularization (tune later if needed)
    "lambda_l2": 0.0,                 # L2 regularization
    "verbosity": -1,                  # Suppress warnings
    "seed": SEED,
    "deterministic": True,
    # "device": "gpu",                  # Use GPU
    # "gpu_platform_id": 0,
    # "gpu_device_id": 0,
}


In [23]:
params_cb = {
    'iterations': 100_000,                  # same as num_boost_round
    'learning_rate': 0.01,                  # same as XGBoost
    'depth': 5,                             # corresponds to max_depth
    'subsample': 0.9279560370175621,        # same meaning
    'colsample_bylevel': 0.5332228504482441, # closest CatBoost equivalent to colsample_bytree
    'loss_function': 'RMSE',                # equivalent to "reg:squarederror"
    'eval_metric': 'RMSE',
    'random_seed': SEED,
    # 'task_type': 'GPU',                     # same as "device": "cuda"
    'early_stopping_rounds': 200,
    'use_best_model': True,
    'verbose': 200,
    'bootstrap_type': 'Bernoulli',          # aligns with subsample usage
    'grow_policy': 'SymmetricTree'          # default, similar to XGBoost depth control
}


In [25]:
oof_preds0 = np.zeros(len(train))
test_preds0 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds0[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds0 += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05856	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:27:57] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04658	valid-rmse:0.05582
[240]	train-rmse:0.04518	valid-rmse:0.05603
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05744	valid-rmse:0.06158


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:27:57] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04697	valid-rmse:0.06104
[340]	train-rmse:0.04239	valid-rmse:0.06107
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05749	valid-rmse:0.06135


...

[200]	train-rmse:0.04816	valid-rmse:0.05554
[225]	train-rmse:0.04726	valid-rmse:0.05558
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05852	valid-rmse:0.05522


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:27:59] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04799	valid-rmse:0.05572
[256]	train-rmse:0.04597	valid-rmse:0.05579
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05333


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:27:59] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04873	valid-rmse:0.05352
[257]	train-rmse:0.04693	valid-rmse:0.05391


In [26]:
train['oof_tap']=pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/oof_tabm_plus_origcol_tuned.csv')['accident_risk']
test['oof_tap']=pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/test_tabm_plus_origcol_tuned.csv')['accident_risk']
train['oof_tap2']=pd.read_csv('/kaggle/input/road-risk-single-ydf/YDF_oof.csv')['YDF']
test['oof_tap2']=pd.read_csv('/kaggle/input/road-risk-single-ydf/YDF_test.csv')['YDF']
train['nn']=pd.read_csv('/kaggle/input/s5e10-nn-stacking-baseline/oof_nn_ensemble.csv')['accident_risk']
test['nn']=pd.read_csv('/kaggle/input/s5e10-nn-stacking-baseline/test_nn_ensemble.csv')['accident_risk']
train['seed']=pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv')['accident_risk']
test['seed']=pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv')['accident_risk']
train['deep']=pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv')['accident_risk']
test['deep']=pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv')['accident_risk']
train['gulon']=pd.read_csv('/kaggle/input/autogulon-0-5539-without-blinding/oof_autogluon_experiment8.csv')['oof_prediction']
test['gulon']=pd.read_csv('/kaggle/input/autogulon-0-5539-without-blinding/autogluon_experiment8.csv')['accident_risk']
train['hill']=pd.read_csv('/kaggle/input/hillclimb/oof.csv')['0']
test['hill']=pd.read_csv('/kaggle/input/hillclimb/submission.csv')['accident_risk']



In [27]:
# # https://www.kaggle.com/code/masayakawamata/s5e10-how-to-improve-your-cv-score
# import optuna
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
#     }
#     params["objective"]="reg:squarederror"
#     params['n_estimators'] = 100000
#     params['learning_rate'] = 0.01
#     params['device'] = 'cpu'
#     params['seed'] = SEED
#     params['verbosity'] = 0
#     # params['early_stopping_rounds'] = 100
#     # params['enable_categorical'] = True
    
#     scores = []

#     oof_preds = np.zeros(len(train))
     
#     kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
#     for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
 
#         X_train = train.iloc[train_idx][FEATURES+TE+['oof_tap','oof_tap2'] ].copy()
#         y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
        
#         X_valid = train.iloc[val_idx][FEATURES+TE+['oof_tap','oof_tap2'] ].copy()
#         y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
#         y_valid2 = train.iloc[val_idx]['y'].values
        
#         X_test = test[FEATURES+TE+['oof_tap','oof_tap2'] ].copy()
             
#         dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
#         dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
     
#         model = xgb.train(
#             params=params,
#             dtrain=dtrain,
#             num_boost_round=100_000,
#             evals=[(dtrain, "train"), (dval, "valid")],
#             early_stopping_rounds=200,
#             verbose_eval=False
#         )
    
#         oof_preds[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
 
#     overall_oof_score = np.sqrt( np.mean( (train[TARGET] - oof_preds)**2. ) )
    
#     return overall_oof_score

# N_TRIALS = 120
# study = optuna.create_study(direction='minimize')

# print(f"--- Starting Optuna Optimization for {N_TRIALS} trials ---")
# study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# print("\n" + "="*50)
# print("OPTUNA OPTIMIZATION COMPLETED")
# print("="*50)

# print(f"Number of finished trials: {len(study.trials)}")
# print("Best trial:")
# best_trial = study.best_trial

# print(f"  Value (OOF RMSE): {best_trial.value:.5f}")
# print("  Params: ")
# for key, value in best_trial.params.items():
#     print(f"    {key}: {value}")

# print("\n--- Top 5 Best Trials (based on OOF RMSE) ---")
# trials_df = study.trials_dataframe().sort_values(by='value', ascending=True)

# for i in range(min(5, len(trials_df))):
#     res = trials_df.iloc[i]
#     print(f"\nRank {i+1}:")
#     print(f"  OOF RMSE: {res['value']:.5f}")
#     params_dict = res.filter(regex='^params_').to_dict()
#     params_dict = {k.replace('params_', ''): v for k, v in params_dict.items()}
#     print(f"  Params: {params_dict}")

# best_params_from_study = study.best_params
# print("\n--- Best Parameters Found ---")
# print(best_params_from_study)

In [28]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['oof_tap']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['oof_tap']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['oof_tap']].copy()
    y_test = test['y'].values

    # LightGBM dataset format
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with callback-based early stopping
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # OOF predictions (add back the base 'y')
    oof_preds[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Test predictions (averaged across folds)
    test_preds += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0505672	valid's rmse: 0.0551934
Early stopping, best iteration is:
[48]	train's rmse: 0.0558266	valid's rmse: 0.0547635
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0504341	valid's rmse: 0.0613218
Early stopping, best iteration is:
[129]	train's rmse: 0.0521421	valid's rmse: 0.0611241
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0508192	valid's rmse: 0.0618729
Early stopping, best iteration is:
[49]	train's rmse: 0.0551291	valid's rmse: 0.0611487
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0496362	valid's rmse: 0.0641011
[40

In [29]:
# XGBoost
oof_preds_ = np.zeros(len(train))
test_preds_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['oof_tap'] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['oof_tap'] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['oof_tap'] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds_[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds_ += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05857	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:09] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04642	valid-rmse:0.05512
[223]	train-rmse:0.04563	valid-rmse:0.05513
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05743	valid-rmse:0.06156


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:10] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04714	valid-rmse:0.06099
[357]	train-rmse:0.04201	valid-rmse:0.06104
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05748	valid-rmse:0.06132


...

[200]	train-rmse:0.04818	valid-rmse:0.05473
[224]	train-rmse:0.04730	valid-rmse:0.05475
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05851	valid-rmse:0.05521


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:11] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04766	valid-rmse:0.05606
[217]	train-rmse:0.04705	valid-rmse:0.05609
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05335


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:11] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04856	valid-rmse:0.05285
[266]	train-rmse:0.04630	valid-rmse:0.05323


In [30]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds__ = np.zeros(len(train))
test_preds__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['oof_tap']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['oof_tap']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE +['oof_tap']].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586191	test: 0.0548543	best: 0.0548543 (0)	total: 53.3ms	remaining: 1h 28m 46s
200:	learn: 0.0538290	test: 0.0546170	best: 0.0544113 (116)	total: 230ms	remaining: 1m 54s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05441129841
bestIteration = 116

Shrink model to first 117 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574859	test: 0.0615758	best: 0.0615758 (0)	total: 948us	remaining: 1m 34s
200:	learn: 0.0531261	test: 0.0613653	best: 0.0613322 (84)	total: 172ms	remaining: 1m 25s
400:	learn: 0.0499935	test: 0.0615802	best: 0.0613284 (218)	total: 350ms	remaining: 1m 26s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06132840195
bestIteration = 218

Shrink model to first 219 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575466	test: 0.0613609	best: 0.0613609 (0)	total: 1.22m

# CV Score
The first CV score below is our XGB model which has improved upon the optimal original data solution (by training on residuals). The second CV score below is using optimal original data solution only.

In [31]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds2 = np.zeros(len(train))
test_preds2 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['oof_tap2', 'oof_tap', 'nn', 'gulon']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['oof_tap2', 'oof_tap', 'nn', 'gulon']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['oof_tap2', 'oof_tap', 'nn', 'gulon']].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds2[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds2 += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586075	test: 0.0548472	best: 0.0548472 (0)	total: 1.66ms	remaining: 2m 46s
200:	learn: 0.0540879	test: 0.0544474	best: 0.0544062 (184)	total: 212ms	remaining: 1m 45s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05440616617
bestIteration = 184

Shrink model to first 185 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574798	test: 0.0615745	best: 0.0615745 (0)	total: 1.4ms	remaining: 2m 19s
200:	learn: 0.0532058	test: 0.0614413	best: 0.0613339 (133)	total: 209ms	remaining: 1m 43s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.0613339191
bestIteration = 133

Shrink model to first 134 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575424	test: 0.0613478	best: 0.0613478 (0)	total: 1.24ms	remaining: 2m 3s
200:	learn: 0.0533459	test: 0.0610826	best: 0.0610356 (111)	total: 216ms	rem

In [32]:
# Xgboost
oof_preds2__ = np.zeros(len(train))
test_preds2__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['oof_tap2',"oof_tap","nn","gulon"] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['oof_tap2',"oof_tap","nn","gulon"] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['oof_tap2',"oof_tap","nn","gulon"] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds2__[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds2__ += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05854	valid-rmse:0.05485


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:21] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04614	valid-rmse:0.05488
[260]	train-rmse:0.04405	valid-rmse:0.05515
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05743	valid-rmse:0.06156


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:21] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04685	valid-rmse:0.06144
[330]	train-rmse:0.04226	valid-rmse:0.06157
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05746	valid-rmse:0.06136


...

[200]	train-rmse:0.04793	valid-rmse:0.05387
[396]	train-rmse:0.04111	valid-rmse:0.05453
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05849	valid-rmse:0.05521


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:23] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04733	valid-rmse:0.05648
[208]	train-rmse:0.04699	valid-rmse:0.05651
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05877	valid-rmse:0.05334


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:24] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04849	valid-rmse:0.05286
[344]	train-rmse:0.04358	valid-rmse:0.05379


In [33]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds2_ = np.zeros(len(train))
test_preds2_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE +['oof_tap2',"oof_tap","nn","gulon"]].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['oof_tap2',"oof_tap","nn","gulon"]].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['oof_tap2',"oof_tap","nn","gulon"]].copy()
    y_test = test['y'].values

    # LightGBM dataset format
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with callback-based early stopping
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # OOF predictions (add back the base 'y')
    oof_preds2_[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Test predictions (averaged across folds)
    test_preds2_ += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502058	valid's rmse: 0.0551495
Early stopping, best iteration is:
[62]	train's rmse: 0.0550347	valid's rmse: 0.0546732
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0503033	valid's rmse: 0.0612251
Early stopping, best iteration is:
[142]	train's rmse: 0.0517153	valid's rmse: 0.061053
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0506321	valid's rmse: 0.0616389
Early stopping, best iteration is:
[47]	train's rmse: 0.0552425	valid's rmse: 0.0610291
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0495314	valid's rmse: 0.06402
Early 

In [34]:
m = np.sqrt( np.mean( (oof_preds2 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds2)
m = np.sqrt( np.mean( (train.y.values - train[TARGET].values)**2. ) )
print(f" Baseline CV RMSE = {m}")

 Overall CV RMSE = 0.057562280258509985
 Baseline CV RMSE = 0.058166805124071685


In [35]:
# XGBoost
oof_preds3 = np.zeros(len(train))
test_preds3 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['nn'] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['nn'] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['nn'] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds3[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds3 += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05857	valid-rmse:0.05483


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:26] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04616	valid-rmse:0.05537
[207]	train-rmse:0.04590	valid-rmse:0.05539
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05744	valid-rmse:0.06158


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:27] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04714	valid-rmse:0.06140
[324]	train-rmse:0.04308	valid-rmse:0.06155
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05746	valid-rmse:0.06134


...

[200]	train-rmse:0.04822	valid-rmse:0.05504
[224]	train-rmse:0.04731	valid-rmse:0.05510
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05849	valid-rmse:0.05523


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:28] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04779	valid-rmse:0.05591
[233]	train-rmse:0.04667	valid-rmse:0.05604
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05336


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:29] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04862	valid-rmse:0.05319
[265]	train-rmse:0.04646	valid-rmse:0.05345


In [36]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds3_ = np.zeros(len(train))
test_preds3_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE +['nn']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE +['nn']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE +['nn']].copy()
    y_test = test['y'].values

    # LightGBM dataset format
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with callback-based early stopping
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # OOF predictions (add back the base 'y')
    oof_preds3_[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Test predictions (averaged across folds)
    test_preds3_ += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0503578	valid's rmse: 0.0553585
Early stopping, best iteration is:
[35]	train's rmse: 0.0565117	valid's rmse: 0.0547002
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0505308	valid's rmse: 0.0612887
Early stopping, best iteration is:
[118]	train's rmse: 0.0524677	valid's rmse: 0.0609945
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0508766	valid's rmse: 0.0618872
Early stopping, best iteration is:
[49]	train's rmse: 0.0551472	valid's rmse: 0.0612111
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0496112	valid's rmse: 0.0640771
[40

In [37]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds3__ = np.zeros(len(train))
test_preds3__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['nn']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['nn']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['nn']].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds3__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds3__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586191	test: 0.0548543	best: 0.0548543 (0)	total: 1ms	remaining: 1m 40s
200:	learn: 0.0538889	test: 0.0546015	best: 0.0544237 (120)	total: 179ms	remaining: 1m 28s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05442370785
bestIteration = 120

Shrink model to first 121 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574859	test: 0.0615758	best: 0.0615758 (0)	total: 1.13ms	remaining: 1m 52s
200:	learn: 0.0531202	test: 0.0613315	best: 0.0612872 (84)	total: 171ms	remaining: 1m 25s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06128715085
bestIteration = 84

Shrink model to first 85 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575466	test: 0.0613609	best: 0.0613609 (0)	total: 979us	remaining: 1m 37s
200:	learn: 0.0532137	test: 0.0612671	best: 0.0611422 (67)	total: 175ms	remainin

In [38]:
m = np.sqrt( np.mean( (oof_preds3 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds3)
m = np.sqrt( np.mean( (train.y.values - train[TARGET].values)**2. ) )
print(f" Baseline CV RMSE = {m}")

 Overall CV RMSE = 0.057579865623523296
 Baseline CV RMSE = 0.058166805124071685


In [39]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds4 = np.zeros(len(train))
test_preds4 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    # Define train/valid/test splits
    X_train = train.iloc[train_idx][FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_test = test['y'].values

    # Create LightGBM datasets
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with early stopping and logging
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # Predict OOF (add back base y)
    oof_preds4[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Predict Test (average across folds)
    test_preds4 += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502287	valid's rmse: 0.0554823
Early stopping, best iteration is:
[30]	train's rmse: 0.0567005	valid's rmse: 0.054825
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0504603	valid's rmse: 0.0616233
Early stopping, best iteration is:
[57]	train's rmse: 0.0545407	valid's rmse: 0.0612286
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0508771	valid's rmse: 0.0615689
Early stopping, best iteration is:
[54]	train's rmse: 0.0550148	valid's rmse: 0.0611315
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0497389	valid's rmse: 0.06391
[400]	t

In [40]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds4__ = np.zeros(len(train))
test_preds4__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['seed', 'nn', 'gulon']].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds4__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds4__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586289	test: 0.0548506	best: 0.0548506 (0)	total: 1.1ms	remaining: 1m 50s
200:	learn: 0.0540381	test: 0.0546495	best: 0.0545572 (73)	total: 200ms	remaining: 1m 39s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.0545571787
bestIteration = 73

Shrink model to first 74 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574612	test: 0.0615657	best: 0.0615657 (0)	total: 1.12ms	remaining: 1m 52s
200:	learn: 0.0533274	test: 0.0614691	best: 0.0614157 (82)	total: 192ms	remaining: 1m 35s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06141568551
bestIteration = 82

Shrink model to first 83 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575304	test: 0.0613502	best: 0.0613502 (0)	total: 1.1ms	remaining: 1m 49s
200:	learn: 0.0532733	test: 0.0612179	best: 0.0611713 (70)	total: 202ms	remaining:

In [41]:
# XGBoost
oof_preds4_ = np.zeros(len(train))
test_preds4_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['seed', 'nn', 'gulon'] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['seed', 'nn', 'gulon'] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['seed', 'nn', 'gulon'] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds4_[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds4_ += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05857	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:39] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04621	valid-rmse:0.05538
[218]	train-rmse:0.04552	valid-rmse:0.05548
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05743	valid-rmse:0.06158


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:40] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04698	valid-rmse:0.06117
[358]	train-rmse:0.04180	valid-rmse:0.06160
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05746	valid-rmse:0.06135


...

[200]	train-rmse:0.04815	valid-rmse:0.05418
[379]	train-rmse:0.04167	valid-rmse:0.05471
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05850	valid-rmse:0.05523


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:42] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04756	valid-rmse:0.05597
[222]	train-rmse:0.04658	valid-rmse:0.05596
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05336


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:42] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04868	valid-rmse:0.05301
[307]	train-rmse:0.04515	valid-rmse:0.05373


In [42]:
m = np.sqrt( np.mean( (oof_preds4 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds4)
m = np.sqrt( np.mean( (train.y.values - train[TARGET].values)**2. ) )
print(f" Baseline CV RMSE = {m}")

 Overall CV RMSE = 0.0577296104234518
 Baseline CV RMSE = 0.058166805124071685


In [43]:
# XGBoost
oof_preds5 = np.zeros(len(train))
test_preds5 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['deep',"nn","gulon"] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['deep',"nn","gulon"] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['deep',"nn","gulon"] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds5[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds5 += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05857	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:43] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04619	valid-rmse:0.05531
[222]	train-rmse:0.04533	valid-rmse:0.05542
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05743	valid-rmse:0.06158


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:43] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04684	valid-rmse:0.06140
[284]	train-rmse:0.04394	valid-rmse:0.06166
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05746	valid-rmse:0.06135


...

[200]	train-rmse:0.04823	valid-rmse:0.05432
[311]	train-rmse:0.04404	valid-rmse:0.05470
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05850	valid-rmse:0.05523


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:45] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04734	valid-rmse:0.05611
[223]	train-rmse:0.04633	valid-rmse:0.05613
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05336


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:45] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04862	valid-rmse:0.05280
[289]	train-rmse:0.04577	valid-rmse:0.05333


In [44]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds5__ = np.zeros(len(train))
test_preds5__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['deep',"nn","gulon"] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['deep',"nn","gulon"] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['deep',"nn","gulon"] ].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds5__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds5__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586325	test: 0.0548553	best: 0.0548553 (0)	total: 891us	remaining: 1m 29s
200:	learn: 0.0539779	test: 0.0546789	best: 0.0544992 (75)	total: 202ms	remaining: 1m 40s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05449919041
bestIteration = 75

Shrink model to first 76 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574660	test: 0.0615663	best: 0.0615663 (0)	total: 1.05ms	remaining: 1m 44s
200:	learn: 0.0533223	test: 0.0614764	best: 0.0614134 (82)	total: 200ms	remaining: 1m 39s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06141344804
bestIteration = 82

Shrink model to first 83 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575304	test: 0.0613502	best: 0.0613502 (0)	total: 1.16ms	remaining: 1m 56s
200:	learn: 0.0532736	test: 0.0611188	best: 0.0610956 (130)	total: 201ms	remaini

In [45]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds5_ = np.zeros(len(train))
test_preds5_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    # Define train/valid/test splits
    X_train = train.iloc[train_idx][FEATURES + TE + ['deep',"nn","gulon"]].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['deep',"nn","gulon"]].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['deep',"nn","gulon"]].copy()
    y_test = test['y'].values

    # Create LightGBM datasets
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with early stopping and logging
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # Predict OOF (add back base y)
    oof_preds5_[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Predict Test (average across folds)
    test_preds5_ += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.050164	valid's rmse: 0.0555426
Early stopping, best iteration is:
[16]	train's rmse: 0.0575079	valid's rmse: 0.0548167
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0504043	valid's rmse: 0.0616346
Early stopping, best iteration is:
[57]	train's rmse: 0.0545649	valid's rmse: 0.0612738
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0508201	valid's rmse: 0.061445
Early stopping, best iteration is:
[58]	train's rmse: 0.0548846	valid's rmse: 0.0611043
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.049567	valid's rmse: 0.0638803
[400]	t

In [46]:
m = np.sqrt( np.mean( (oof_preds5 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds5)
m = np.sqrt( np.mean( (train.y.values - train[TARGET].values)**2. ) )
print(f" Baseline CV RMSE = {m}")

 Overall CV RMSE = 0.05746371631421285
 Baseline CV RMSE = 0.058166805124071685


In [47]:
# XGBoost
oof_preds6 = np.zeros(len(train))
test_preds6 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+["gulon","nn"] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['gulon',"nn"] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['gulon',"nn"] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds6[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds6 += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05857	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:51] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04614	valid-rmse:0.05546
[216]	train-rmse:0.04566	valid-rmse:0.05547
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05743	valid-rmse:0.06159


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:51] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04683	valid-rmse:0.06127
[290]	train-rmse:0.04384	valid-rmse:0.06125
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05745	valid-rmse:0.06135


...

[200]	train-rmse:0.04806	valid-rmse:0.05432
[330]	train-rmse:0.04325	valid-rmse:0.05474
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05850	valid-rmse:0.05523


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:53] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04766	valid-rmse:0.05624
[219]	train-rmse:0.04696	valid-rmse:0.05644
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05880	valid-rmse:0.05334


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:28:53] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04880	valid-rmse:0.05310
[348]	train-rmse:0.04358	valid-rmse:0.05373


In [48]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds6__ = np.zeros(len(train))
test_preds6__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ["gulon","nn"] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ["gulon","nn"] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ["gulon","nn"] ].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds6__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds6__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586205	test: 0.0548516	best: 0.0548516 (0)	total: 1ms	remaining: 1m 40s
200:	learn: 0.0539507	test: 0.0547576	best: 0.0545821 (147)	total: 191ms	remaining: 1m 34s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05458208079
bestIteration = 147

Shrink model to first 148 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574916	test: 0.0615823	best: 0.0615823 (0)	total: 1.27ms	remaining: 2m 6s
200:	learn: 0.0531370	test: 0.0615428	best: 0.0613013 (79)	total: 198ms	remaining: 1m 38s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06130129908
bestIteration = 79

Shrink model to first 80 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575364	test: 0.0613676	best: 0.0613676 (0)	total: 1.04ms	remaining: 1m 43s
200:	learn: 0.0532106	test: 0.0611554	best: 0.0611078 (142)	total: 200ms	remaini

In [49]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds6_ = np.zeros(len(train))
test_preds6_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    # Define train/valid/test splits
    X_train = train.iloc[train_idx][FEATURES + TE + ["gulon","nn"]].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ["gulon","nn"]].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ["gulon","nn"]].copy()
    y_test = test['y'].values

    # Create LightGBM datasets
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with early stopping and logging
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # Predict OOF (add back base y)
    oof_preds6_[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Predict Test (average across folds)
    test_preds6_ += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502265	valid's rmse: 0.0556447
Early stopping, best iteration is:
[7]	train's rmse: 0.0581824	valid's rmse: 0.0547367
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502685	valid's rmse: 0.0615247
Early stopping, best iteration is:
[104]	train's rmse: 0.0527883	valid's rmse: 0.0611642
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0508423	valid's rmse: 0.0616294
Early stopping, best iteration is:
[61]	train's rmse: 0.0547031	valid's rmse: 0.0610144
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0494519	valid's rmse: 0.0638421
[400

In [50]:
m = np.sqrt( np.mean( (oof_preds6 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds6)

 Overall CV RMSE = 0.057461296989213644


In [51]:
# XGBoost
oof_preds7= np.zeros(len(train))
test_preds7 = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#"*25)
    print(f"### Fold {fold+1} ###")
    print("#"*25)

    X_train = train.iloc[train_idx][FEATURES+TE+['hill',"nn" ] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']
    
    X_valid = train.iloc[val_idx][FEATURES+TE+['hill',"nn" ] ].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values
    
    X_test = test[FEATURES+TE+['hill',"nn" ] ].copy()
    y_test = test['y'].values
        
    dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100_000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200
    )

    oof_preds7[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
    test_preds7 += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS

#########################
### Fold 1 ###
#########################
[0]	train-rmse:0.05856	valid-rmse:0.05484


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:29:00] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04616	valid-rmse:0.05548
[215]	train-rmse:0.04565	valid-rmse:0.05555
#########################
### Fold 2 ###
#########################
[0]	train-rmse:0.05744	valid-rmse:0.06159


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:29:00] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04678	valid-rmse:0.06111
[290]	train-rmse:0.04370	valid-rmse:0.06114
#########################
### Fold 3 ###
#########################
[0]	train-rmse:0.05745	valid-rmse:0.06135


...

[200]	train-rmse:0.04799	valid-rmse:0.05439
[324]	train-rmse:0.04329	valid-rmse:0.05470
#########################
### Fold 6 ###
#########################
[0]	train-rmse:0.05850	valid-rmse:0.05523


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:29:02] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04766	valid-rmse:0.05640
[220]	train-rmse:0.04691	valid-rmse:0.05649
#########################
### Fold 7 ###
#########################
[0]	train-rmse:0.05879	valid-rmse:0.05336


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [17:29:02] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)


[200]	train-rmse:0.04864	valid-rmse:0.05341
[312]	train-rmse:0.04475	valid-rmse:0.05393


In [52]:
# lightgbm
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import KFold

oof_preds7_ = np.zeros(len(train))
test_preds7_ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    # Define train/valid/test splits
    X_train = train.iloc[train_idx][FEATURES + TE + ['hill',"nn" ]].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['hill',"nn" ]].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['hill',"nn" ]].copy()
    y_test = test['y'].values

    # Create LightGBM datasets
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

    # Train with early stopping and logging
    model = lgb.train(
        params=params_l,
        train_set=lgb_train,
        num_boost_round=100_000,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    # Predict OOF (add back base y)
    oof_preds7_[val_idx] = model.predict(X_valid, num_iteration=model.best_iteration) + y_valid2

    # Predict Test (average across folds)
    test_preds7_ += (model.predict(X_test, num_iteration=model.best_iteration) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502178	valid's rmse: 0.0553002
Early stopping, best iteration is:
[7]	train's rmse: 0.058179	valid's rmse: 0.054733
#########################
### Fold 2 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0502712	valid's rmse: 0.0613931
Early stopping, best iteration is:
[107]	train's rmse: 0.0526968	valid's rmse: 0.0610668
#########################
### Fold 3 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.050842	valid's rmse: 0.0615365
Early stopping, best iteration is:
[61]	train's rmse: 0.0547012	valid's rmse: 0.0610011
#########################
### Fold 4 ###
#########################
Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 0.0495166	valid's rmse: 0.0638721
[400]	t

In [53]:
# catboost
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold

oof_preds7__ = np.zeros(len(train))
test_preds7__ = np.zeros(len(test))

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print("#" * 25)
    print(f"### Fold {fold + 1} ###")
    print("#" * 25)

    X_train = train.iloc[train_idx][FEATURES + TE + ['hill',"nn" ] ].copy()
    y_train = train.iloc[train_idx][TARGET] - train.iloc[train_idx]['y']

    X_valid = train.iloc[val_idx][FEATURES + TE + ['hill',"nn" ]].copy()
    y_valid = train.iloc[val_idx][TARGET] - train.iloc[val_idx]['y']
    y_valid2 = train.iloc[val_idx]['y'].values

    X_test = test[FEATURES + TE + ['hill',"nn" ] ].copy()
    y_test = test['y'].values

    # Define categorical features (if you have any)
    cat_features = [col for col in X_train.columns if X_train[col].dtype == 'object']

    # Prepare CatBoost Pool
    train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_features)
    test_pool  = Pool(X_test, cat_features=cat_features)

    # Define parameters (you can adapt from your XGBoost params)
  

    model = CatBoostRegressor(**params_cb)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=200
    )

    oof_preds7__[val_idx] = model.predict(valid_pool) + y_valid2
    test_preds7__ += (model.predict(test_pool) + y_test) / FOLDS


#########################
### Fold 1 ###
#########################
0:	learn: 0.0586206	test: 0.0548518	best: 0.0548518 (0)	total: 1.04ms	remaining: 1m 44s
200:	learn: 0.0539961	test: 0.0548296	best: 0.0546378 (56)	total: 190ms	remaining: 1m 34s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.05463780253
bestIteration = 56

Shrink model to first 57 iterations.
#########################
### Fold 2 ###
#########################
0:	learn: 0.0574917	test: 0.0615824	best: 0.0615824 (0)	total: 1.19ms	remaining: 1m 59s
200:	learn: 0.0531361	test: 0.0615144	best: 0.0613023 (79)	total: 192ms	remaining: 1m 35s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.06130228521
bestIteration = 79

Shrink model to first 80 iterations.
#########################
### Fold 3 ###
#########################
0:	learn: 0.0575364	test: 0.0613676	best: 0.0613676 (0)	total: 1.1ms	remaining: 1m 49s
200:	learn: 0.0532250	test: 0.0611757	best: 0.0610945 (97)	total: 200ms	remainin

In [54]:
m = np.sqrt( np.mean( (oof_preds7 - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")
np.save(f"oof",oof_preds7)

 Overall CV RMSE = 0.05758483738406903


In [56]:
a=pd.DataFrame(oof_preds2,columns=["z"])
b=pd.DataFrame(oof_preds,columns=["y"])
c=pd.DataFrame(oof_preds0,columns=["x"])
d=pd.DataFrame(oof_preds3,columns=["m"])
e=pd.DataFrame(oof_preds4,columns=["e"])
f=pd.DataFrame(oof_preds5,columns=["f"])
g=pd.DataFrame(oof_preds6,columns=["g"])
h=pd.DataFrame(oof_preds7,columns=["h"])
aa=pd.DataFrame(oof_preds2_,columns=["zz"])
bb=pd.DataFrame(oof_preds_,columns=["yy"])
dd=pd.DataFrame(oof_preds3_,columns=["mm"])
ee=pd.DataFrame(oof_preds4_,columns=["ee"])
ff=pd.DataFrame(oof_preds5_,columns=["ff"])
gg=pd.DataFrame(oof_preds6_,columns=["gg"])
hh=pd.DataFrame(oof_preds7_,columns=["hh"])
aaa=pd.DataFrame(oof_preds2__,columns=["zzz"])
bbb=pd.DataFrame(oof_preds__,columns=["yyy"])
ddd=pd.DataFrame(oof_preds3__,columns=["mmm"])
eee=pd.DataFrame(oof_preds4__,columns=["eee"])
fff=pd.DataFrame(oof_preds5__,columns=["fff"])
ggg=pd.DataFrame(oof_preds6__,columns=["ggg"])
hhh=pd.DataFrame(oof_preds7__,columns=["hhh"])

oof_pp=pd.concat([a,b,c,d,e,f,g,h,
                  aa,bb,dd,ee,ff,gg,hh,
                  aaa,bbb,ddd,eee,fff,ggg,hhh,
                  train['oof_tap'],train['oof_tap2'],train['nn'],train['seed'],train['deep'],train['gulon'],train['hill']],axis=1)

oof_pp['mean_h'] = 0.1*oof_pp['x']+0.2*oof_pp['y']+0.2*oof_pp['z']+0.3*oof_pp['m']+0.1*oof_pp['e']+0.1*oof_pp['f']
oof_pp['mean_g'] = 0.2*oof_pp['e']+0.2*oof_pp['f']+0.6*oof_pp['g']
oof_pp['mean'] = oof_pp.mean(axis=1)
oof_pp['std'] = oof_pp.std(axis=1)
oof_pp.isna().sum()
oof_pp.to_csv('oof_pp.csv',index=False)

z           0
y           0
x           0
m           0
e           0
f           0
g           0
h           0
zz          0
yy          0
mm          0
ee          0
ff          0
gg          0
hh          0
zzz         0
yyy         0
mmm         0
eee         0
fff         0
ggg         0
hhh         0
oof_tap     0
oof_tap2    0
nn          0
seed        0
deep        0
gulon       0
hill        0
mean_h      0
mean_g      0
mean        0
std         0
dtype: int64

In [60]:
a=pd.DataFrame(test_preds2,columns=["z"])
b=pd.DataFrame(test_preds,columns=["y"])
c=pd.DataFrame(test_preds0,columns=["x"])
d=pd.DataFrame(test_preds3,columns=["m"])
e=pd.DataFrame(test_preds4,columns=["e"])
f=pd.DataFrame(test_preds5,columns=["f"])
g=pd.DataFrame(test_preds6,columns=["g"])
h=pd.DataFrame(test_preds7,columns=["h"])
aa=pd.DataFrame(test_preds2_,columns=["zz"])
bb=pd.DataFrame(test_preds_,columns=["yy"])
dd=pd.DataFrame(test_preds3_,columns=["mm"])
ee=pd.DataFrame(test_preds4_,columns=["ee"])
ff=pd.DataFrame(test_preds5_,columns=["ff"])
gg=pd.DataFrame(test_preds6_,columns=["gg"])
hh=pd.DataFrame(test_preds7_,columns=["hh"])
aaa=pd.DataFrame(test_preds2__,columns=["zzz"])
bbb=pd.DataFrame(test_preds__,columns=["yyy"])
ddd=pd.DataFrame(test_preds3__,columns=["mmm"])
eee=pd.DataFrame(test_preds4__,columns=["eee"])
fff=pd.DataFrame(test_preds5__,columns=["fff"])
ggg=pd.DataFrame(test_preds6__,columns=["ggg"])
hhh=pd.DataFrame(test_preds7__,columns=["hhh"])

pred_pred=pd.concat([a,b,c,d,e,f,g,h,
                  aa,bb,dd,ee,ff,gg,hh,
                  aaa,bbb,ddd,eee,fff,ggg,hhh,
                  test['oof_tap'],test['oof_tap2'],test['nn'],test['seed'],test['deep'],test['gulon'],test['hill']],axis=1)

pred_pred['mean_h'] =  0.1*pred_pred['x']+0.2*pred_pred['y']+0.2*pred_pred['z']+0.3*pred_pred['m']+0.1*pred_pred['e']+0.1*pred_pred['f']
pred_pred['mean_g'] = 0.2*pred_pred['e']+0.2*pred_pred['f']+0.6*pred_pred['g']
pred_pred['mean'] = pred_pred.mean(axis=1)
pred_pred['std'] = pred_pred.std(axis=1)
pred_pred.isna().sum()
pred_pred.to_csv('pred_pred.csv',index=False)

z           0
y           0
x           0
m           0
e           0
f           0
g           0
h           0
zz          0
yy          0
mm          0
ee          0
ff          0
gg          0
hh          0
zzz         0
yyy         0
mmm         0
eee         0
fff         0
ggg         0
hhh         0
oof_tap     0
oof_tap2    0
nn          0
seed        0
deep        0
gulon       0
hill        0
mean_h      0
mean_g      0
mean        0
std         0
dtype: int64

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

N_SPLITS = 7
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# FEATURES = [col for col in oof_df.columns if col not in ['id',TARGET]]

X = oof_pp.copy()
y = train[TARGET]


oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(pred_pred))

SEED = [32, 12, 377, 485, 5900, 2392, 3948, 189, 304598, 38950]

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f'---Fold {fold+1}/{N_SPLITS}---')

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test = pred_pred.copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    for seed in SEED:
        np.random.seed(seed)
        tf.random.set_seed(seed)
        model = Sequential([
            Input(shape=(X_train_scaled.shape[1],)), 
            Dense(64, activation='relu'),            
            Dense(32, activation='relu'),            
            Dense(1)                                 
        ])
    
        model.compile(optimizer='adam', loss='mean_squared_error')
    
        early_stopping = EarlyStopping(
            monitor='val_loss', 
            patience=20,       
            restore_best_weights=True 
        )
    
        model.fit(X_train_scaled, y_train,
                  validation_data=(X_val_scaled, y_val),
                  epochs=200,       
                  batch_size=512,
                  callbacks=[early_stopping],
                  verbose=0         
                 )
    
        val_preds = model.predict(X_val_scaled).flatten() 
        oof_preds[val_idx] += val_preds / len(SEED)
    
        test_preds += model.predict(X_test_scaled).flatten() / len(SEED)

    fold_rmse = mean_squared_error(y_val, oof_preds[val_idx], squared=False)
    print(f"Fold {fold+1} RMSE: {fold_rmse}")

test_preds /= N_SPLITS

overall_oof_rmse = mean_squared_error(y, oof_preds, squared=False)
print(f"\nOverall OOF RMSE: {overall_oof_rmse:.5f}")

---Fold 1/7---


2025-10-30 17:31:45.204233: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [ ]:
m = np.sqrt( np.mean( (oof_preds - train[TARGET].values)**2. ) )
print(f" Overall CV RMSE = {m}")

In [ ]:
 
# residuals = np.abs(train[TARGET] - train['y'])  # الريزيدوال (الخطأ المطلق)

# # ==== الخطوة 3: اختيار العينات الأفضل من test ====
# # نختار فقط العينات اللي النموذج متأكد منها (أقل خطأ)
# threshold = np.percentile(residuals, 5)  # مثلاً نحتفظ بـ 30% الأقل خطأ
# mask = residuals <= threshold

# X_pseudo =  test[mask]
# y_pseudo = pd.Series(test_preds)[mask]  # نستخدم القيم المتوقعة كـ pseudo labels

# print(f"تم اختيار {len(X_pseudo)} عينة من test لإضافتها إلى التدريب.")
# trainp = pd.concat([X_pseudo,y_pseudo],axis=1).reset_index(drop=True)
# train = pd.concat([train,trainp],axis=0).reset_index(drop=True)




In [ ]:

# oof_preds_r = np.zeros(len(train))
# test_preds_r = np.zeros(len(test))

# kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
# for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
#     print("#"*25)
#     print(f"### Fold {fold+1} ###")
#     print("#"*25)

#     X_train = train.iloc[train_idx][FEATURES+TE].copy()
#     y_train = residuals_.iloc[train_idx]
    
#     X_valid = train.iloc[val_idx][FEATURES+TE].copy()
#     y_valid = residuals_.iloc[val_idx] 
#     y_valid2 = pd.Series(oof_preds).iloc[val_idx].values
    
#     X_test = test[FEATURES+TE].copy()
#     y_test = test_preds
        
#     dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
#     dval   = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
#     dtest  = xgb.DMatrix(X_test, enable_categorical=True)

#     model = xgb.train(
#         params=params,
#         dtrain=dtrain,
#         num_boost_round=100_000,
#         evals=[(dtrain, "train"), (dval, "valid")],
#         early_stopping_rounds=200,
#         verbose_eval=200
#     )

#     oof_preds_r[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration + 1)) +y_valid2
#     test_preds_r += (model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) +y_test)/ FOLDS
    

In [ ]:
# m = np.sqrt( np.mean( (oof_preds_r - train[TARGET].values)**2. ) )
# print(f" Overall CV RMSE = {m}")
# m = np.sqrt( np.mean( (train.y.values - train[TARGET].values)**2. ) )
# print(f" Baseline CV RMSE = {m}")

# OOF EDA
We plot true vs predicted below. Discussion from @tilii7 (Kaggle user Tilii) about this plot is [here][1]

[1]: https://www.kaggle.com/competitions/playground-series-s5e10/discussion/610422

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(train[TARGET].values,oof_preds,s=0.25)
plt.plot([0,1],[0,1],'--',color='black')
plt.title("True vs Predicted")
plt.xlabel("True Target")
plt.ylabel("Predicted Target")
plt.show()

# XGB Feature Importance

In [ ]:
# plt.rcParams["figure.dpi"] = 280      
# fig, ax = plt.subplots(figsize=(15, 12))

# xgb.plot_importance(
#     model,
#     max_num_features=150,
#     importance_type="gain",
#     ax=ax,
#     show_values=False,                
#     grid=False
# )

# ax.set_title("XGB Feature Importances", fontsize=18)
# ax.tick_params(axis="both", labelsize=12)
# fig.tight_layout()
# plt.show()

# Create Submission CSV

In [ ]:
sub = pd.read_csv("/kaggle/input/playground-series-s5e10/sample_submission.csv")  
sub['accident_risk'] = test_preds
sub.to_csv("submission.csv",index=False)
sub.head()

# Test Pred EDA

In [ ]:
plt.hist(sub['accident_risk'],bins=100)
plt.title("Histogram of Test Preds")
plt.show()